<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/12_trajectory_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12 · Integration tests: trajectory evals

An agent can produce a perfect answer by an absurd route: calling the same tool nine times,
skipping the policy check and guessing correctly, or looking up an unrelated order first.

Final-response evals pass all of those. **Trajectory evals are how you test the path.**

**New in this lesson:** `create_trajectory_match_evaluator` with its four match modes, tool-args
matching, `create_trajectory_llm_as_judge`, and choosing a metric that is not brittle.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "agentevals~=0.0.9" \
  "openevals~=0.2.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-12-trajectory"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. What a trajectory is

A list of messages, in order, including every tool call. `agentevals` works with the OpenAI
message format, so capture a real run and convert it.

In [ ]:
import json

from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=(
        "You are a customer support agent.\n"
        "Look up the order before answering questions about it.\n"
        "Check the refund policy before promising any remedy."
    ),
)


def to_trajectory(messages) -> list[dict]:
    """Convert LangChain messages to the OpenAI-style dicts agentevals expects."""
    out = []
    for m in messages:
        role = getattr(m, "type", None)
        if role == "human":
            out.append({"role": "user", "content": m.text})
        elif role == "ai":
            entry = {"role": "assistant", "content": m.text or ""}
            calls = getattr(m, "tool_calls", None) or []
            if calls:
                entry["tool_calls"] = [
                    {
                        "id": tc.get("id") or "call",
                        # json.dumps, not str(): agentevals parses these arguments as JSON.
                        "function": {"name": tc["name"],
                                     "arguments": json.dumps(tc.get("args", {}))},
                    }
                    for tc in calls
                ]
            out.append(entry)
        elif role == "tool":
            out.append({"role": "tool", "content": str(m.content)[:200],
                        "tool_call_id": getattr(m, "tool_call_id", "call")})
    return out


QUESTION = "Order 1047 - the laptop stand wobbles. Can they get a refund?"
result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
trajectory = to_trajectory(result["messages"])

for step in trajectory:
    calls = [tc["function"]["name"] for tc in step.get("tool_calls", [])]
    print(f"{step['role']:10} {calls if calls else step['content'][:60]!r}")

---

## 2. Strict match, and why it disappoints

`strict` requires the same tool calls in the same order. Start here to see the problem.

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

strict = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="ignore",   # compare the shape of the path, not the arguments
)

# Use the run we just captured as the reference.
reference = trajectory

print("against itself:", strict(outputs=trajectory, reference_outputs=reference))

In [ ]:
# Now a second run of the SAME question. Nothing about the agent changed.
second = to_trajectory(
    agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})["messages"]
)

print("run 1 tools:", [tc["function"]["name"] for s in trajectory for tc in s.get("tool_calls", [])])
print("run 2 tools:", [tc["function"]["name"] for s in second for tc in s.get("tool_calls", [])])
print()
print("strict match:", strict(outputs=second, reference_outputs=reference))

If the second run checked the policy before looking up the order — a completely reasonable
ordering — strict match fails it. The agent did nothing wrong.

### 🧠 Checkpoint

The agent behaved correctly and the test went red.

Name the bug: is it in the agent, or in the metric?

<details><summary>Show answer</summary>

**In the metric.**

Strict match encodes a claim you did not actually mean to make: *"there is exactly one correct
order for these tool calls."* For most agents that is false. Looking up the order first and
reading the policy first are both fine; the requirement is that **both happen before the
answer**.

This matters beyond the immediate annoyance. A test that fails on correct behaviour gets muted,
then ignored, then deleted — and it takes the real regressions with it. A brittle test is worse
than no test, because it consumes the attention you would have spent on a real one.

Use strict only where order genuinely is the contract: authenticate before charging, validate
before writing, check permissions before disclosing.

</details>

---

## 3. Four modes

| Mode | Passes when | Use for |
|---|---|---|
| `strict` | same calls, same order | order is genuinely the contract |
| `unordered` | same set of calls, any order | "these steps must all happen" |
| `subset` | actual ⊆ reference | "do not do anything extra" |
| `superset` | actual ⊇ reference | "at minimum, do these" |

In [ ]:
modes = ["strict", "unordered", "subset", "superset"]

for mode in modes:
    ev = create_trajectory_match_evaluator(
        trajectory_match_mode=mode, tool_args_match_mode="ignore",
    )
    print(f"{mode:11} {ev(outputs=second, reference_outputs=reference)['score']}")

**`superset` is the workhorse.** It expresses the requirement you usually mean: *the agent must
at least check the order and the policy; anything else it does is its own business.* That
survives model upgrades and prompt tweaks while still catching a skipped safety step.

In [ ]:
# A minimal reference: the two calls that MUST happen, whatever else does.
required = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": "",
     "tool_calls": [{"id": "a", "function": {"name": "lookup_order", "arguments": "{}"}}]},
    {"role": "tool", "content": "...", "tool_call_id": "a"},
    {"role": "assistant", "content": "",
     "tool_calls": [{"id": "b", "function": {"name": "get_refund_policy", "arguments": "{}"}}]},
    {"role": "tool", "content": "...", "tool_call_id": "b"},
    {"role": "assistant", "content": "answer"},
]

must_check_both = create_trajectory_match_evaluator(
    trajectory_match_mode="superset", tool_args_match_mode="ignore",
)

print("did the agent check both?", must_check_both(outputs=second, reference_outputs=required))

---

## 4. Tool arguments — where this gets real

Calling `lookup_order` is not enough; calling it with the *right order id* is the point.
`tool_args_match_mode` has the same four settings, applied to arguments.

In [ ]:
wrong_order = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": "",
     "tool_calls": [{"id": "a", "function": {"name": "lookup_order",
                                             "arguments": '{"order_id": "1042"}'}}]},
    {"role": "tool", "content": "...", "tool_call_id": "a"},
    {"role": "assistant", "content": "answer"},
]

right_order = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": "",
     "tool_calls": [{"id": "a", "function": {"name": "lookup_order",
                                             "arguments": '{"order_id": "1047"}'}}]},
    {"role": "tool", "content": "...", "tool_call_id": "a"},
    {"role": "assistant", "content": "answer"},
]

ignoring = create_trajectory_match_evaluator(
    trajectory_match_mode="unordered", tool_args_match_mode="ignore")
exacting = create_trajectory_match_evaluator(
    trajectory_match_mode="unordered", tool_args_match_mode="exact")

print("args ignored, wrong order id:", ignoring(outputs=wrong_order, reference_outputs=right_order)["score"])
print("args exact,   wrong order id:", exacting(outputs=wrong_order, reference_outputs=right_order)["score"])

In [ ]:
# Per-tool overrides: be exact where it matters, relaxed where it does not.
mixed = create_trajectory_match_evaluator(
    trajectory_match_mode="superset",
    tool_args_match_mode="ignore",              # default: relaxed
    tool_args_match_overrides={
        "lookup_order": "exact",                # but the order id must be right
        "search_tickets": "ignore",             # phrasing of a search query is free
    },
)

print(mixed(outputs=wrong_order, reference_outputs=right_order))

That combination — **loose on the path, strict on the arguments that carry meaning** — is the
one to reach for by default.

---

## 5. When you cannot write a reference

Sometimes there is no single right path, and you want a judgement about whether the route was
*sensible*. That is a trajectory judge.

In [ ]:
from agentevals.trajectory.llm import create_trajectory_llm_as_judge

TRAJECTORY_RUBRIC = """
Grade this customer support agent's TRAJECTORY, not its final answer.

<trajectory>
{outputs}
</trajectory>

Score true only if ALL of these hold:
  - it looked up the order before making claims about that order
  - it consulted the refund policy before offering any remedy
  - it did not call the same tool repeatedly with identical arguments
  - each step plausibly follows from the previous one

Score false if it guessed, skipped the policy check, or looped.
"""

path_judge = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_RUBRIC, model=MODEL, feedback_key="sensible_path",
)

print("real run:  ", path_judge(outputs=second))

In [ ]:
# A trajectory that reaches a plausible answer without doing the work.
guessing = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": "Since it's been over 30 days, we can only offer a repair."},
]

# And one that loops.
looping = [
    {"role": "user", "content": QUESTION},
    *[
        item
        for i in range(4)
        for item in (
            {"role": "assistant", "content": "",
             "tool_calls": [{"id": f"c{i}", "function": {"name": "lookup_order",
                                                         "arguments": '{"order_id": "1047"}'}}]},
            {"role": "tool", "content": "Order 1047 ...", "tool_call_id": f"c{i}"},
        )
    ],
    {"role": "assistant", "content": "Repair only."},
]

print("guessed:", path_judge(outputs=guessing))
print()
print("looped: ", path_judge(outputs=looping))

Both of those produce a *correct final answer*. Only a trajectory eval can tell you the first
one got there by luck and the second burned four identical tool calls doing it.

---

## 6. Graph trajectories

`agentevals` can also evaluate the **node path** through a LangGraph graph — the layer you saw
in lesson 09 — rather than the message list. Useful when your control flow is the thing under
test.

```python
from agentevals.graph_trajectory.utils import extract_langgraph_trajectory_from_thread
from agentevals.graph_trajectory.strict import graph_trajectory_strict_match
from agentevals.graph_trajectory.llm import create_graph_trajectory_llm_as_judge

trajectory = extract_langgraph_trajectory_from_thread(
    agent, {"configurable": {"thread_id": "some-thread"}},
)
```

Reach for it when you have written custom graph structure. For a standard agent loop the message
trajectory is the more informative view, because the interesting variation is in tool calls
rather than node transitions.

### 🧠 Checkpoint

You want to catch one specific regression: **the agent sometimes answers refund questions
without reading the policy.**

Pick the loosest metric that still catches it, and say why each tighter option is worse.

<details><summary>Show answer</summary>

**`superset`, args ignored, with a reference containing only the `get_refund_policy` call.**

That asserts exactly one thing: the policy was consulted. Nothing about order, nothing about
what else happened, nothing about arguments (`get_refund_policy` takes none).

Why the tighter options are worse:

- **`strict`** fails whenever the agent reorders two harmless calls. Red on correct behaviour.
- **`unordered`** requires the sets to match exactly, so adding a legitimate `search_tickets`
  call fails it. You would be forbidding improvement.
- **`exact` args** adds nothing here and creates a fragile coupling to argument formatting.

The general rule for this whole lesson: **assert the smallest thing that would have caught the
bug.** Every additional constraint is a future false failure, and false failures get tests
deleted.

</details>

### ✍️ Exercise

An agent is skipping a required verification step: it should call `search_tickets` to check for
a prior complaint before escalating a repeat issue, and sometimes it does not.

Write the metric that:

1. **fails** a trajectory that escalates without ever calling `search_tickets`
2. **passes** a trajectory that calls it, regardless of ordering or search wording
3. **passes** a trajectory that does extra legitimate work as well

Test it against all three trajectories to prove it discriminates correctly.

<details><summary>Show a solution</summary>

```python
from agentevals.trajectory.match import create_trajectory_match_evaluator

# Assert one thing only: search_tickets happened.
required = [
    {"role": "user", "content": "Repeat problem with order 1042 — escalate?"},
    {"role": "assistant", "content": "",
     "tool_calls": [{"id": "s", "function": {"name": "search_tickets", "arguments": "{}"}}]},
    {"role": "tool", "content": "...", "tool_call_id": "s"},
    {"role": "assistant", "content": "answer"},
]

checked_history = create_trajectory_match_evaluator(
    trajectory_match_mode="superset",
    tool_args_match_mode="ignore",   # search wording is free
)

def traj(*tools):
    out = [{"role": "user", "content": "Repeat problem with order 1042 — escalate?"}]
    for i, name in enumerate(tools):
        out.append({"role": "assistant", "content": "",
                    "tool_calls": [{"id": f"t{i}",
                                    "function": {"name": name, "arguments": "{}"}}]})
        out.append({"role": "tool", "content": "...", "tool_call_id": f"t{i}"})
    out.append({"role": "assistant", "content": "answer"})
    return out

cases = {
    "skips search (should FAIL)":       traj("lookup_order"),
    "searches, other order (PASS)":     traj("search_tickets", "lookup_order"),
    "searches + extra work (PASS)":     traj("lookup_order", "search_tickets", "get_refund_policy"),
}

for label, t in cases.items():
    print(f"{label:34} {checked_history(outputs=t, reference_outputs=required)['score']}")
```

</details>

---

## 📌 Key takeaways

- Trajectory evals catch process bugs that a correct final answer hides — guessing, looping, skipped checks.
- `strict` encodes "there is exactly one right order", which is usually false. It is rarely the right default.
- `superset` is the workhorse: assert the steps that must happen, allow everything else.
- Tool-args matching is where trajectory evals become real — loose on the path, exact on arguments that carry meaning.
- `tool_args_match_overrides` lets you be strict per tool instead of globally.
- A trajectory judge handles "was this route sensible?" when no single reference path exists.
- A test that goes red on correct behaviour gets muted, then deleted — brittleness is a real cost.
- Assert the smallest thing that would have caught the bug.

---

## ➡️ Next

**[13 · Evals in CI](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/13_evals_in_ci.ipynb)**

You have evaluators for steps and for paths. Running them by hand is a demo — next they run on
every pull request.